# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [28]:
import json
from pathlib import Path
import pandas as pd
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode
import numpy as np

In [29]:
df = pd.read_csv("C:/Users/isabe/Perspectiva_dados/PLN/df_quotes.csv")
print(len(df))
print(df)

100
                                                frase               autor  \
0   “The world as we have created it is a process ...     Albert Einstein   
1   “It is our choices, Harry, that show what we t...        J.K. Rowling   
2   “There are only two ways to live your life. On...     Albert Einstein   
3   “The person, be it gentleman or lady, who has ...         Jane Austen   
4   “Imperfection is beauty, madness is genius and...      Marilyn Monroe   
..                                                ...                 ...   
95  “You never really understand a person until yo...          Harper Lee   
96  “You have to write the book that wants to be w...   Madeleine L'Engle   
97  “Never tell the truth to people who are not wo...          Mark Twain   
98        “A person's a person, no matter how small.”           Dr. Seuss   
99  “... a mind needs books as a sword needs a whe...  George R.R. Martin   

                                                 tags  
0    ['change',

In [30]:
#
#noticias = []
#
#for arquivo in sorted(Path("data").glob("*.json")):
#    with arquivo.open(encoding="utf-8") as f:
#        noticias.append(json.load(f))
#
#df = pd.DataFrame(noticias)
#
#print(f"Antes: {len(df)} noticias")
#
#df = df.drop_duplicates(subset="url").reset_index(drop=True)
#
#print(f"Depois: {len(df)} noticias")
#
#df.head()

## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [31]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["frase_limpa"] = df["frase"].apply(limpar_texto)

df[["frase", "frase_limpa"]].head()

,frase,frase_limpa
0,“The world as we have created it is a process ...,the world as we have created it is a process o...
1,"“It is our choices, Harry, that show what we t...",it is our choices harry that show what we trul...
2,“There are only two ways to live your life. On...,there are only two ways to live your life one ...
3,"“The person, be it gentleman or lady, who has ...",the person be it gentleman or lady who has not...
4,"“Imperfection is beauty, madness is genius and...",imperfection is beauty madness is genius and i...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [51]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("english")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["frase_limpa"].apply(remover_stopwords)
df["frase_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["frase_limpa", "tokens_sem_stopwords", "frase_sem_stopwords"]].head()

df

,frase,autor,tags,frase_limpa,tokens_sem_stopwords,frase_sem_stopwords
0,“The world as we have created it is a process ...,Albert Einstein,"['change', 'deep-thoughts', 'thinking', 'world']",the world as we have created it is a process o...,"[world, created, process, thinking, cannot, ch...",world created process thinking cannot changed ...
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"['abilities', 'choices']",it is our choices harry that show what we trul...,"[choices, harry, show, truly, far, abilities]",choices harry show truly far abilities
2,“There are only two ways to live your life. On...,Albert Einstein,"['inspirational', 'life', 'live', 'miracle', '...",there are only two ways to live your life one ...,"[two, ways, live, life, one, though, nothing, ...",two ways live life one though nothing miracle ...
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"['aliteracy', 'books', 'classic', 'humor']",the person be it gentleman or lady who has not...,"[person, gentleman, lady, pleasure, good, nove...",person gentleman lady pleasure good novel must...
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"['be-yourself', 'inspirational']",imperfection is beauty madness is genius and i...,"[imperfection, beauty, madness, genius, better...",imperfection beauty madness genius better abso...
...,...,...,...,...,...,...
95,“You never really understand a person until yo...,Harper Lee,['better-life-empathy'],you never really understand a person until you...,"[never, really, understand, person, consider, ...",never really understand person consider things...
96,“You have to write the book that wants to be w...,Madeleine L'Engle,"['books', 'children', 'difficult', 'grown-ups'...",you have to write the book that wants to be wr...,"[write, book, wants, written, book, difficult,...",write book wants written book difficult grown ...
97,“Never tell the truth to people who are not wo...,Mark Twain,['truth'],never tell the truth to people who are not wor...,"[never, tell, truth, people, worthy]",never tell truth people worthy
98,"“A person's a person, no matter how small.”",Dr. Seuss,['inspirational'],a person s a person no matter how small,"[person, person, matter, small]",person person matter small


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [33]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["frase_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,000,10,abilities,absolutely,absurdities,accept,across,act,actually,admiration,...,worthy,would,wrap,write,writing,written,wrote,wrung,year,youer
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [34]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

2 colunas removidas


,abilities,absolutely,absurdities,accept,across,act,actually,admiration,admit,adventure,...,worthy,would,wrap,write,writing,written,wrote,wrung,year,youer
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [35]:
metadados = df[["frase", "autor", "tags"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

In [9]:
df_final.to_csv("df_final.csv", index=False)

In [36]:
df_final.head(10)

,frase,autor,tags,bow_abilities,bow_absolutely,bow_absurdities,bow_accept,bow_across,bow_act,bow_actually,...,bow_worthy,bow_would,bow_wrap,bow_write,bow_writing,bow_written,bow_wrote,bow_wrung,bow_year,bow_youer
0,“The world as we have created it is a process ...,Albert Einstein,"['change', 'deep-thoughts', 'thinking', 'world']",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"['abilities', 'choices']",1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,“There are only two ways to live your life. On...,Albert Einstein,"['inspirational', 'life', 'live', 'miracle', '...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"['aliteracy', 'books', 'classic', 'humor']",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"['be-yourself', 'inspirational']",0,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,“Try not to become a man of success. Rather be...,Albert Einstein,"['adulthood', 'success', 'value']",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,“It is better to be hated for what you are tha...,André Gide,"['life', 'love']",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"['edison', 'failure', 'inspirational', 'paraph...",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,['misattributed-eleanor-roosevelt'],0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"['humor', 'obvious', 'simile']",0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [37]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 556


bow_love     23
bow_one      15
bow_never    13
bow_life     12
bow_think    12
bow_make     10
bow_like      9
bow_good      8
bow_live      7
bow_see       7
dtype: int64

In [38]:
frequencia_palavras.tail(10)

bow_worthy        1
bow_dependence    1
bow_wrap          1
bow_devil         1
bow_writing       1
bow_written       1
bow_wrote         1
bow_wrung         1
bow_year          1
bow_youer         1
dtype: int64

In [39]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

love      15
one       13
never     11
life      10
like       9
think      8
people     7
good       7
know       7
make       7
dtype: int64

In [40]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["frase", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,frase,palavras_unicas
64,“Not all those who wander are lost.”,2
6,“It is better to be hated for what you are tha...,3
56,"“He's like a drug for you, Bella.”",3
47,“Only in the darkness can you see the stars.”,3
57,“There is no friend as loyal as a book.”,3
39,“It is never too late to be what you might hav...,3
36,“We read to know we're not alone.”,3
59,“Life isn't about finding yourself. Life is ab...,3
98,"“A person's a person, no matter how small.”",3
43,"“If you can make a woman laugh, you can make h...",4


## TF-IDF

O Bag of Words transforma textos em numeros contando quantas vezes cada palavra aparece em cada noticia. Essa representacao e simples e util, mas tem uma limitacao importante: palavras muito frequentes podem parecer importantes apenas porque aparecem muito.

O TF-IDF e uma forma de dar peso para as palavras tentando responder a duas perguntas ao mesmo tempo:

- Esta palavra aparece bastante nesta noticia?
- Esta palavra e rara no conjunto de noticias?

A ideia e que uma palavra deve receber peso alto quando aparece bastante em uma noticia, mas nao aparece em todas as noticias. Assim, o TF-IDF ajuda a destacar palavras mais caracteristicas de cada documento.

Como ja criamos o df_final, temos uma matriz Bag of Words pronta nas colunas que comecam com bow_.

Para calcular o TF-IDF manualmente com pandas, o caminho e:

- Separar apenas as colunas bow_ do df_final.
- Remover o prefixo bow_ dos nomes das colunas, para trabalhar diretamente com as palavras. (df_bow no nosso caso)
- Calcular o total de palavras de cada noticia somando as linhas da matriz.
- Calcular o TF dividindo cada valor da linha pelo total da propria linha.
- Calcular em quantas noticias cada palavra aparece.
- Calcular o IDF usando log(N/DF)
- Multiplicar a matriz de TF pelo vetor de IDF.

O resultado final e uma nova matriz: cada linha continua sendo uma noticia, cada coluna continua sendo uma palavra, mas agora os valores sao pesos TF-IDF em vez de contagens brutas.

In [12]:
df_bow

,abilities,absolutely,absurdities,accept,across,act,actually,admiration,admit,adventure,...,worthy,would,wrap,write,writing,written,wrote,wrung,year,youer
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,2,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,0,0,0,0,0,0,0,0,0,0,...,0,0,0,2,0,1,0,0,0,0
97,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
98,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [41]:
# Total de palavras
TP = []
TP = df_bow.sum(axis=1)
TP

0      9
1      6
2     11
3      9
4      9
      ..
95    13
96    10
97     5
98     4
99     8
Length: 100, dtype: int64

In [42]:
TF = df_bow.div(TP, axis=0)
TF

,abilities,absolutely,absurdities,accept,across,act,actually,admiration,admit,adventure,...,worthy,would,wrap,write,writing,written,wrote,wrung,year,youer
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.166667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.222222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.2,0.0,0.1,0.0,0.0,0.0,0.0
97,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
98,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
df_ = (df_bow > 0).sum(axis=0)
df_

abilities      1
absolutely     1
absurdities    1
accept         1
across         1
              ..
written        1
wrote          1
wrung          1
year           1
youer          1
Length: 556, dtype: int64

In [44]:
N = df_bow.shape[0]
N = len(df_bow)
IDF = np.log(N / df_)
df_idf = pd.DataFrame({
    "documentos_com_o_termo": df_,
    "idf": IDF
}).sort_values("idf", ascending=False)

df_idf.head(10)

,documentos_com_o_termo,idf
youer,1,4.60517
abilities,1,4.60517
absolutely,1,4.60517
absurdities,1,4.60517
accept,1,4.60517
across,1,4.60517
act,1,4.60517
actually,1,4.60517
admiration,1,4.60517
admit,1,4.60517


In [45]:

frase = 0

print(df_final.loc[frase, "frase"])

TF.loc[frase].sort_values(ascending=False).head(10)

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”


thinking    0.222222
cannot      0.111111
changing    0.111111
created     0.111111
changed     0.111111
process     0.111111
without     0.111111
world       0.111111
act         0.000000
actually    0.000000
Name: 0, dtype: float64

In [46]:

df_idf.tail(10)

,documentos_com_o_termo,idf
people,7,2.659260
know,7,2.659260
make,7,2.659260
good,7,2.659260
think,8,2.525729
like,9,2.407946
life,10,2.302585
never,11,2.207275
one,13,2.040221
love,15,1.897120


In [47]:

TF_IDF = TF.mul(IDF, axis=1)

TF_IDF.head()

,abilities,absolutely,absurdities,accept,across,act,actually,admiration,admit,adventure,...,worthy,would,wrap,write,writing,written,wrote,wrung,year,youer
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.767528,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,1.023371,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [48]:
TF_IDF.loc[0].sort_values(ascending=False).head(10)

thinking    0.779235
cannot      0.511686
changing    0.511686
created     0.511686
changed     0.511686
process     0.511686
without     0.332859
world       0.312601
act         0.000000
actually    0.000000
Name: 0, dtype: float64

In [49]:
TF_IDF_pref = TF_IDF.add_prefix("tfidf_").reset_index(drop=True)

DF_TFIDF = pd.concat([metadados, TF_IDF_pref], axis=1)

DF_TFIDF.head()

,frase,autor,tags,tfidf_abilities,tfidf_absolutely,tfidf_absurdities,tfidf_accept,tfidf_across,tfidf_act,tfidf_actually,...,tfidf_worthy,tfidf_would,tfidf_wrap,tfidf_write,tfidf_writing,tfidf_written,tfidf_wrote,tfidf_wrung,tfidf_year,tfidf_youer
0,“The world as we have created it is a process ...,Albert Einstein,"['change', 'deep-thoughts', 'thinking', 'world']",0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"['abilities', 'choices']",0.767528,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,“There are only two ways to live your life. On...,Albert Einstein,"['inspirational', 'life', 'live', 'miracle', '...",0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"['aliteracy', 'books', 'classic', 'humor']",0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"['be-yourself', 'inspirational']",0.000000,1.023371,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
from pathlib import Path
# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[["frase", "autor", "tags", "frase_limpa", "frase_sem_stopwords"]]
df_completo.to_csv("Frases_completo.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv("df_bow.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
DF_TFIDF.to_csv("TF_IDF.csv", index=False)
